In [59]:
# =============================================================================
# TAREA 1: Exploracion y Analisis de Calidad de Datos
# Curso: Ingenieria de Datos con Python
# =============================================================================

import pandas as pd
import json
import os
from datetime import datetime
from IPython.display import display

# Carpeta de salida para evidencias de Tarea 1
os.makedirs('homeworks/output/tarea_1', exist_ok=True)

In [62]:
# Carga de datos 
csv_path = 'data/raw/ventas.csv'

if not os.path.exists(csv_path):
    raise FileNotFoundError(f'No existe el archivo: {csv_path}')

df_ventas = pd.read_csv(csv_path)
print('Archivo cargado:', csv_path)
print('Forma del archivo:', df_ventas.shape)
display(df_ventas.head(5))

Archivo cargado: data/raw/ventas.csv
Forma del archivo: (20000, 31)


,id_venta,fecha,año,mes,dia_semana,semana_numero,producto,categoria,cantidad,precio_unitario,...,cliente_antiguedad_dias,vendedor_id,vendedor_experiencia_meses,promocion_aplicada,cupon_usado,puntuacion_cliente,tiempo_entrega_dias,fecha_registro_cliente,acumulado_mes_producto,ranking_ciudad_mes
0,1,2024-12-03,2024,12,Tuesday,49,Disco Duro,Electrónica,46,976,...,1,Sofia,6.0,CyberMonday,No,3,3,2024-12-02,35916.8,1.0
1,2,08-26-2024,2024,8,Monday,35,Teclado,NaN,57,1309,...,219,Laura,24.0,Verano,Sí,3,15,2024-01-20,64913.31,1.0
2,3,2024-08-21,2024,8,Wednesday,34,Monitor,Accesorios,8,727,...,7,Sofia,6.0,CyberMonday,False,1,3,2024-08-14,5699.68,2.0
3,4,2024-07-07,2024,7,Sunday,27,Mouse,Almacenamiento,15,705,...,99,Ana,36.0,Sin promoción,False,1,12,2024-03-30,5499.0,1.0
4,5,2023-05-15,2023,5,Monday,20,NaN,Electrónica,56,1500,...,184,Sofia,6.0,NaN,True,2,5,2022-11-12,84000.0,1.0


In [10]:
# =============================================================================
# Analisis de calidad (requisitos de Tarea 1)
# =============================================================================

print('===== 1) Totales generales =====')
total_registros = len(df_ventas)
total_unicos = len(df_ventas.drop_duplicates())
print('Total de registros:', total_registros)
print('Total de registros unicos (sin duplicados):', total_unicos)

print('\n===== 2) Nulos por columna =====')
nulos = df_ventas.isna().sum()
porcentaje_nulos = (nulos / total_registros * 100).round(2)
tabla_nulos = pd.DataFrame({
    'columna': df_ventas.columns,
    'nulos': nulos.values,
    'porcentaje_nulos': porcentaje_nulos.values
}).sort_values('porcentaje_nulos', ascending=False)
display(tabla_nulos)

print('\n===== 3) Tipos actuales vs esperados =====')
tipos_actuales = df_ventas.dtypes.astype(str)

expected_by_column = {
    'id_venta': 'int64',
    'fecha': 'datetime64[ns]',
    'cantidad': 'int64',
    'precio_unitario': 'float64',
    'monto_total': 'float64',
    'cliente_id': 'object'
}

tipos_esperados = [expected_by_column.get(col, 'N/A') for col in df_ventas.columns]
tabla_tipos = pd.DataFrame({
    'columna': df_ventas.columns,
    'tipo_actual': tipos_actuales.values,
    'tipo_esperado': tipos_esperados
})
display(tabla_tipos)

print('\n===== 4) Problemas especificos =====')

amount_candidates = ['amount', 'monto_total', 'precio_unitario']
date_candidates = ['date', 'fecha', 'fecha_transaccion']

amount_col = next((c for c in amount_candidates if c in df_ventas.columns), None)
date_col = next((c for c in date_candidates if c in df_ventas.columns), None)

problemas = {
    'columna_monto': amount_col,
    'columna_fecha': date_col,
    'montos_negativos': 0,
    'montos_string_con_$': 0,
    'fechas_formato_incorrecto': 0
}

if amount_col is not None:
    monto_raw = df_ventas[amount_col].astype(str)
    problemas['montos_string_con_$'] = int(monto_raw.str.contains(r'\$', regex=True, na=False).sum())

    monto_num = pd.to_numeric(
        monto_raw.str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    )
    problemas['montos_negativos'] = int((monto_num < 0).sum())
else:
    print('No se encontro columna de monto para validar negativos y $')

if date_col is not None:
    fechas = pd.to_datetime(df_ventas[date_col], errors='coerce', dayfirst=False)
    problemas['fechas_formato_incorrecto'] = int(fechas.isna().sum())
else:
    print('No se encontro columna de fecha para validar formato')

print('Montos negativos:', problemas['montos_negativos'])
print('Montos como string (con $):', problemas['montos_string_con_$'])
print('Fechas en formato incorrecto:', problemas['fechas_formato_incorrecto'])

print('\n===== 5) Resumen en formato tabla =====')
tabla_resumen = pd.DataFrame({
    'metrica': [
        'total_registros',
        'total_registros_unicos',
        'total_columnas',
        'total_nulos',
        'montos_negativos',
        'montos_string_con_$',
        'fechas_formato_incorrecto'
    ],
    'valor': [
        int(total_registros),
        int(total_unicos),
        int(df_ventas.shape[1]),
        int(nulos.sum()),
        int(problemas['montos_negativos']),
        int(problemas['montos_string_con_$']),
        int(problemas['fechas_formato_incorrecto'])
    ]
})
display(tabla_resumen)

===== 1) Totales generales =====
Total de registros: 20000
Total de registros unicos (sin duplicados): 20000

===== 2) Nulos por columna =====


,columna,nulos,porcentaje_nulos
17,metodo_pago,6610,33.05
30,ranking_ciudad_mes,4669,23.34
13,margen_beneficio,4078,20.39
18,canal_venta,4058,20.29
7,categoria,4026,20.13
20,categoria_cliente,4008,20.04
14,estado,4004,20.02
25,cupon_usado,3790,18.95
16,region,3387,16.93
15,ciudad,3387,16.93



===== 3) Tipos actuales vs esperados =====


,columna,tipo_actual,tipo_esperado
0,id_venta,int64,int64
1,fecha,object,datetime64[ns]
2,año,int64,N/A
3,mes,int64,N/A
4,dia_semana,object,N/A
5,semana_numero,int64,N/A
6,producto,object,N/A
7,categoria,object,N/A
8,cantidad,object,int64
9,precio_unitario,object,float64



===== 4) Problemas especificos =====
Montos negativos: 2809
Montos como string (con $): 0
Fechas en formato incorrecto: 4000

===== 5) Resumen en formato tabla =====


,metrica,valor
0,total_registros,20000
1,total_registros_unicos,20000
2,total_columnas,31
3,total_nulos,56171
4,montos_negativos,2809
5,montos_string_con_$,0
6,fechas_formato_incorrecto,4000


In [11]:
# =============================================================================
# Exportar reporte JSON de calidad
# =============================================================================

reporte = {
    'fecha_analisis': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'archivo_analizado': csv_path,
    'totales': {
        'registros': int(total_registros),
        'registros_unicos': int(total_unicos),
        'columnas': int(df_ventas.shape[1])
    },
    'nulos_por_columna': tabla_nulos.to_dict(orient='records'),
    'tipos': tabla_tipos.to_dict(orient='records'),
    'problemas_especificos': {
        'columna_monto': problemas['columna_monto'],
        'columna_fecha': problemas['columna_fecha'],
        'montos_negativos': int(problemas['montos_negativos']),
        'montos_string_con_$': int(problemas['montos_string_con_$']),
        'fechas_formato_incorrecto': int(problemas['fechas_formato_incorrecto'])
    },
    'resumen': tabla_resumen.to_dict(orient='records')
}

salida_json = 'homeworks/output/tarea_1/reporte_calidad.json'
with open(salida_json, 'w', encoding='utf-8') as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False)

print('Reporte guardado en:', salida_json)

Reporte guardado en: homeworks/output/tarea_1/reporte_calidad.json
